In [1]:
from pathlib import Path
from io import StringIO
import re
import pandas as pd
from bs4 import BeautifulSoup

source_path = Path('../data/raw/bigwest_2026_women.html')
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

soup = BeautifulSoup(source_path.read_text(encoding='utf-8'), 'html.parser')
for tag in soup.find_all(attrs={'colspan': True}):
    if not str(tag['colspan']).isdigit():
        tag['colspan'] = '1'

table_tags = soup.find_all('table')
tables = pd.read_html(StringIO(str(soup)))
assert len(table_tags) == len(tables), 'HTML/Pandas table counts differ; check table alignment'
print('Tables found:', len(tables))


c:\Users\12094\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Tables found: 69


In [2]:
FIELD_KEYWORDS = ('long jump', 'triple jump', 'high jump', 'pole vault',
                  'shot put', 'discus', 'hammer', 'javelin')
COMBINED_KEYWORDS = ('heptathlon', 'decathlon', 'pentathlon')

def get_event_info(table_tag):
    heading = table_tag.find_previous(['h1', 'h2', 'h3', 'h4', 'h5'])
    while heading:
        text = heading.get_text(' ', strip=True)
        if re.search(r"(?:Women['’]s|Men['’]s)", text, re.I):
            break
        heading = heading.find_previous(['h1', 'h2', 'h3', 'h4', 'h5'])
    if not heading:
        return None, None, None, None
    text = heading.get_text(' ', strip=True)
    event = re.split(r'\b(?:Finals?|Preliminaries|Semi-Finals?|Flight|Heat|Section)\b',
                     text, maxsplit=1, flags=re.I)[0]
    event = re.sub(r"^(?:Women['’]s|Men['’]s)\s+", '', event).strip()
    low = event.lower()
    event_type = ('combined' if any(x in low for x in COMBINED_KEYWORDS)
                  else 'field' if any(x in low for x in FIELD_KEYWORDS)
                  else 'running')
    round_name = ('Flight' if re.search(r'\bFlight\b', text, re.I)
                  else 'Heat' if re.search(r'\bHeat\b', text, re.I)
                  else 'Preliminaries' if re.search(r'\bPreliminaries\b', text, re.I)
                  else 'Finals' if re.search(r'\bFinals?\b', text, re.I)
                  else 'Other')
    return event, event_type, round_name, text

# Main overall results tables for the 2026 women's saved HTML.
# Indices are specific to this file; check them when changing years/genders.
FIELD_SUMMARY_TABLES = {
    'High Jump': 48, 'Pole Vault': 49, 'Long Jump': 50,
    'Triple Jump': 53, 'Shot Put': 56, 'Discus': 59,
    'Hammer': 62, 'Javelin': 65,
}

# Only Triple Jump MARK.2 has been independently checked against the
# previously verified results. Other multi-MARK selections need review.
VERIFIED_FIELD_MARK_COLUMNS = {'Triple Jump': 'MARK.2',
                               'Discus': 'MARK', 'Hammer': 'MARK'}

def is_field_summary_table(df):
    return {'PL', 'NAME', 'TEAM', 'MARK'}.issubset(df.columns)

def get_official_mark_column(df, event_type, round_name, event=None):
    if event_type == 'field':
        cols = [c for c in df.columns if re.fullmatch(r'MARK(?:\.\d+)?', str(c))]
        if not cols:
            return None
        return VERIFIED_FIELD_MARK_COLUMNS.get(event, cols[-1])
    cols = [c for c in df.columns if re.fullmatch(r'TIME(?:\.\d+)?', str(c))]
    if not cols:
        return None
    # Running-time selection is provisional and should be independently checked.
    return cols[0] if round_name in ('Flight', 'Heat', 'Preliminaries') else cols[-1]


In [3]:
# Verify expected overall results tables and the mark columns they contain.
for event, i in FIELD_SUMMARY_TABLES.items():
    table = tables[i]
    mark_cols = [c for c in table.columns if re.fullmatch(r'MARK(?:\.\d+)?', str(c))]
    chosen = get_official_mark_column(table, 'field', 'Other', event)
    status = 'VERIFIED' if event in VERIFIED_FIELD_MARK_COLUMNS else 'REVIEW MARK COLUMN'
    print(f'{event:14s} table={i:2d} chosen={chosen:8s} {status} available={mark_cols}')


High Jump      table=48 chosen=MARK.3   REVIEW MARK COLUMN available=['MARK', 'MARK.1', 'MARK.2', 'MARK.3']
Pole Vault     table=49 chosen=MARK.4   REVIEW MARK COLUMN available=['MARK', 'MARK.1', 'MARK.2', 'MARK.3', 'MARK.4']
Long Jump      table=50 chosen=MARK.3   REVIEW MARK COLUMN available=['MARK', 'MARK.1', 'MARK.2', 'MARK.3']
Triple Jump    table=53 chosen=MARK.2   VERIFIED available=['MARK', 'MARK.1', 'MARK.2']
Shot Put       table=56 chosen=MARK.6   REVIEW MARK COLUMN available=['MARK', 'MARK.1', 'MARK.2', 'MARK.3', 'MARK.4', 'MARK.5', 'MARK.6']
Discus         table=59 chosen=MARK     VERIFIED available=['MARK']
Hammer         table=62 chosen=MARK     VERIFIED available=['MARK']
Javelin        table=65 chosen=MARK.4   REVIEW MARK COLUMN available=['MARK', 'MARK.1', 'MARK.2', 'MARK.3', 'MARK.4']


In [ ]:
def clean_event_table(table, tag, index, gender, field_tables):
    event, event_type, round_name, heading = get_event_info(tag)
    if not event or event_type == 'combined':
        return None
    df = table.copy()
    if event_type == 'field':
        # Use only the main overall results table, not the flight tables.
        if index != field_tables.get(event):
            return None
        if not is_field_summary_table(df):
            return None
    elif not {'PL', 'NAME', 'TEAM'}.issubset(df.columns):
        return None

    mark_col = get_official_mark_column(df, event_type, round_name, event)
    if mark_col is None:
        return None
    df = df.rename(columns={'PL':'place', 'NAME':'athlete_name', 'YEAR':'class_year',
                            'TEAM':'school', 'SC':'points', 'WIND':'wind',
                            'CONV':'converted_mark', mark_col:'mark'})
    ids = []
    for tr in tag.select('tbody tr'):
        link = tr.select_one('a[href*="/athletes/"]')
        m = re.search(r'/athletes/(\d+)', link.get('href', '')) if link else None
        ids.append(m.group(1) if m else None)
    if len(ids) != len(df):
        print(f'SKIP table {index}: athlete-link rows {len(ids)} != parsed rows {len(df)}')
        return None
    df['athlete_id'] = ids
    df = df[df['athlete_id'].notna() & df['athlete_name'].notna() & df['school'].notna()].copy()
    if df.empty:
        return None
    for optional in ('class_year', 'wind', 'converted_mark'):
        if optional not in df: df[optional] = None
    if 'points' not in df: df['points'] = 0
    df['points'] = pd.to_numeric(df['points'], errors='coerce').fillna(0)
    df['place'] = pd.to_numeric(df['place'], errors='coerce')
    df['year'], df['gender'], df['event'] = 2026, 'Women', event
    df['event_type'], df['round'] = event_type, round_name
    df['source_table'], df['source_heading'] = index, heading
    df['mark_source_column'] = mark_col
    return df[['year','gender','event','event_type','round','athlete_id',
               'athlete_name','class_year','school','place','mark','converted_mark',
               'points','wind','source_table','source_heading','mark_source_column']]

parsed = []
field_candidates = set()
field_selected = set()
for i, (table, tag) in enumerate(zip(tables, table_tags)):
    event, event_type, round_name, heading = get_event_info(tag)
    if event_type == 'field':
        field_candidates.add(event)
    try:
        result = clean_event_table(table, tag, i)
        if result is not None:
            parsed.append(result)
            if event_type == 'field':
                field_selected.add(event)
                print(f'FIELD SUMMARY: table {i}, {event}, {len(result)} athletes')
    except Exception as exc:
        print(f'Error on table {i}: {exc}')

missing = sorted(field_candidates - field_selected)
if missing:
    print('WARNING: Missing field events:', missing)
if not missing:
    print('All 8 field events were extracted!')
if not parsed:
    raise ValueError('No result tables parsed. Inspect source HTML and table columns.')
all_results = pd.concat(parsed, ignore_index=True)
print('Parsed rows:', len(all_results))


FIELD SUMMARY: table 48, High Jump, 18 athletes
FIELD SUMMARY: table 49, Pole Vault, 15 athletes
FIELD SUMMARY: table 50, Long Jump, 25 athletes
FIELD SUMMARY: table 53, Triple Jump, 18 athletes
FIELD SUMMARY: table 56, Shot Put, 23 athletes
FIELD SUMMARY: table 59, Discus, 24 athletes
FIELD SUMMARY: table 62, Hammer, 24 athletes
FIELD SUMMARY: table 65, Javelin, 17 athletes
All 8 field events were extracted!
Parsed rows: 688


In [5]:
# Field rows come only from the eight identified main results tables.
# Running results still prioritize finals over prelims and retain prelim-only athletes.
priority = {'Finals': 4, 'Other': 3, 'Preliminaries': 2, 'Flight': 1, 'Heat': 1}
all_results['round_priority'] = all_results['round'].map(priority).fillna(0)
all_results['has_mark'] = all_results['mark'].notna() & ~all_results['mark'].astype(str).str.strip().eq('')
all_results = all_results.sort_values(
    ['round_priority', 'has_mark', 'points', 'source_table'],
    ascending=[False, False, False, True]
)
selected = all_results.drop_duplicates(subset=['year','gender','event','athlete_id'], keep='first').copy()
running_df = selected[selected['event_type']=='running'].copy()
field_df = selected[selected['event_type']=='field'].copy()
assert field_df.apply(lambda r: int(r['source_table']) == FIELD_SUMMARY_TABLES.get(r['event']), axis=1).all(), 'Unexpected field source table'
assert field_df['source_table'].map(lambda i: is_field_summary_table(tables[int(i)])).all(), 'Non-summary field table used'
print('Running rows:', len(running_df), '| Field rows:', len(field_df))


Running rows: 269 | Field rows: 164


In [6]:
# Check source tables, results and totals for each field event.
print('FIELD RESULTS BY EVENT')
display(field_df.groupby('event').agg(entries=('athlete_name','size'),
                                     points=('points','sum'),
                                     source_tables=('source_table', lambda s: sorted(s.unique()))))
for event in ['Triple Jump', 'Javelin', 'Discus']:
    subset = field_df[field_df['event'].str.fullmatch(event, case=False, na=False)]
    print(f'\n{event}: {len(subset)} entries')
    display(subset[['athlete_name','place','mark','converted_mark','points','source_table']].sort_values('place'))
print('\nRUNNING EVENTS')
display(running_df.groupby('event').agg(entries=('athlete_name','size'), points=('points','sum')))

print('\nFIELD MARK COLUMN AUDIT')
display(field_df.groupby('event').agg(
    athletes=('athlete_id', 'nunique'),
    points=('points', 'sum'),
    source_table=('source_table', 'first'),
    mark_column=('mark_source_column', 'first')
))
print('CAUTION: Only Triple Jump, Discus, and Hammer mark columns have been identified from prior checks.')
print('Inspect the other five field-event marks against the official page before model training.')


FIELD RESULTS BY EVENT


,entries,points,source_tables
event,,,
Discus,24,39.0,[59]
Hammer,24,39.0,[62]
High Jump,18,39.0,[48]
Javelin,17,39.0,[65]
Long Jump,25,39.0,[50]
Pole Vault,15,39.0,[49]
Shot Put,23,39.0,[56]
Triple Jump,18,39.0,[53]



Triple Jump: 18 entries


,athlete_name,place,mark,converted_mark,points,source_table
582,Chelsea Aninyei,1.0,13.04m,"42' 9.5""",10.0,53
583,Tamara Aimufia,2.0,13.01m,"42' 8.25""",8.0,53
584,Camryn Harris,3.0,12.86m,"42' 2.25""",6.0,53
585,Kennedy Johnson,4.0,12.79m,"41' 11.5""",5.0,53
586,Tara Wyllie,5.0,12.77m,"41' 10.75""",4.0,53
587,Daneira Pellum,6.0,11.95m,"39' 2.5""",3.0,53
588,Chidiebere Agunanne,7.0,11.93m,"39' 1.75""",2.0,53
589,Grace Sloan,8.0,11.93m,"39' 1.75""",1.0,53
590,Karalyn Cheng,9.0,11.83m,"38' 9.75""",0.0,53
591,Sydney Roquebrun,10.0,11.80m,"38' 8.75""",0.0,53



Javelin: 17 entries


...


RUNNING EVENTS


,entries,points
event,,
"10,000 Meters",23,39.0
100 Hurdles,19,39.0
100 Meters,36,39.0
1500 Meters,23,39.0
200 Meters,34,39.0
3000 Steeplechase,18,39.0
400 Hurdles,22,39.0
400 Meters,29,39.0
5000 Meters,38,39.0



FIELD MARK COLUMN AUDIT


,athletes,points,source_table,mark_column
event,,,,
Discus,24,39.0,59,MARK
Hammer,24,39.0,62,MARK
High Jump,18,39.0,48,MARK.3
Javelin,17,39.0,65,MARK.4
Long Jump,25,39.0,50,MARK.3
Pole Vault,15,39.0,49,MARK.4
Shot Put,23,39.0,56,MARK.6
Triple Jump,18,39.0,53,MARK.2


CAUTION: Only Triple Jump, Discus, and Hammer mark columns have been identified from prior checks.
Inspect the other five field-event marks against the official page before model training.


In [ ]:
# Export only after reviewing the diagnostics above.
running_path = output_dir / 'bigwest_2026_women_running.csv'
field_path = output_dir / 'bigwest_2026_women_field.csv'
running_df.to_csv(running_path, index=False)
field_df.to_csv(field_path, index=False)
print('Saved:', running_path, field_path, sep='\n')


Saved:
..\data\processed\bigwest_2026_women_running.csv
..\data\processed\bigwest_2026_women_field.csv


: 

In [ ]:
running_df

## Quality notes
- Uses the eight identified overall field-results tables (2026 women), not flight tables.
- Triple Jump uses verified `MARK.2`; Discus and Hammer use `MARK`. Other multi-MARK events temporarily use the last `MARK` column and **must be audited** before modeling.
- Running time selection remains heuristic and should be audited separately.
- Field table indices will change for other years/genders.
